In [0]:
# CELL 1: Install Dependencies & Setup Parameters
# The week_ending_date parameter is what makes this pipeline incremental.
%pip install openpyxl
dbutils.widgets.text("base_dir", "/Volumes/workspace/default/spi_data/samples")
dbutils.widgets.text("week_ending_date", "24.09.2026")

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# CELL 2: Read Data via Pandas to handle complex Excel formatting
import pandas as pd
from pyspark.sql.functions import current_timestamp, lit
from datetime import datetime

base_dir = dbutils.widgets.get("base_dir")
report_date = dbutils.widgets.get("week_ending_date")
start_time = datetime.now()

# Read as strings to bypass header corruption
pdf_report = pd.read_excel(f"{base_dir}/3.-SPI-Report-{report_date}.xlsx", engine='openpyxl').astype(str)
pdf_annex = pd.read_excel(f"{base_dir}/Annex_{report_date}.xlsx", engine='openpyxl').astype(str)

# Convert to Spark and add exact load timestamps
df_report = spark.createDataFrame(pdf_report) \
    .withColumn("load_timestamp", current_timestamp()) \
    .withColumn("week_ending_date", lit(report_date))
    
df_annex = spark.createDataFrame(pdf_annex) \
    .withColumn("load_timestamp", current_timestamp()) \
    .withColumn("week_ending_date", lit(report_date))

# Verification: Check what the raw data looks like
print("--- Raw PBS Report Preview ---")
display(df_report.limit(3))

--- Raw PBS Report Preview ---


Annexure-III,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,load_timestamp,week_ending_date
nan,nan,nan,nan,nan,nan,nan,nan,2026-10-07T18:41:52.049Z,24.09.2026
Government of Pakistan,nan,nan,nan,nan,nan,nan,nan,2026-10-07T18:41:52.049Z,24.09.2026
nan,nan,nan,nan,nan,nan,nan,nan,2026-10-07T18:41:52.049Z,24.09.2026


In [0]:
# CELL 3: Idempotent Write to Bronze
# replaceWhere ensures that loading "24.09.2026" twice doesn't double the rows.
import uuid as _uuid
from pyspark.sql.functions import lit

# Align DataFrame schemas with existing Bronze tables
for col_name in df_report.columns:
    if col_name.startswith("Unnamed: "):
        df_report = df_report.withColumnRenamed(col_name, col_name.replace("Unnamed: ", "Unnamed__"))
df_report = df_report.withColumn("source_file", lit(f"3.-SPI-Report-{report_date}"))

for col_name in df_annex.columns:
    if col_name.startswith("Unnamed: "):
        df_annex = df_annex.withColumnRenamed(col_name, col_name.replace("Unnamed: ", "Unnamed__"))
df_annex = df_annex.withColumn("source_file", lit(f"Annex_{report_date}"))

try:
    df_report.write.format("delta").mode("overwrite") \
        .option("replaceWhere", f"week_ending_date = '{report_date}'") \
        .saveAsTable("default.bronze_pbs_report")
        
    df_annex.write.format("delta").mode("overwrite") \
        .option("replaceWhere", f"week_ending_date = '{report_date}'") \
        .saveAsTable("default.bronze_pbs_annex")

    rows = df_report.count() + df_annex.count()
    spark.sql(f"INSERT INTO default.pipeline_execution_logs VALUES ('{_uuid.uuid4()}', 'Raw-to-Bronze-PBS', 'Week_{report_date}', '{start_time}', current_timestamp(), 'Success', {rows}, '')")
    print(f"Success: Wrote {rows} rows to Bronze PBS.")
except Exception as e:
    spark.sql(f"INSERT INTO default.pipeline_execution_logs VALUES ('{_uuid.uuid4()}', 'Raw-to-Bronze-PBS', 'Week_{report_date}', '{start_time}', current_timestamp(), 'Failure', 0, '{str(e)[0:200]}')")
    raise e

Success: Wrote 218 rows to Bronze PBS.
